# Train a simple classification head on embeddings

Data: 4 slices (3 slices for training, 1 slice for testing); Label: domain

Input: embedding

Prediction: domain

Model: Linear(d, 512) → GELU → Dropout(0.2) → Linear(512, C)

## Train

In [51]:
# define params
input_emb_dim: int = 256  # the dimension of input embeddings
num_classes: int = 7  # the # of classification classes
hidden_dim: int = 128  # dim of hidden layer
dropout: float = 0.2  # drop out ratio
lr: float = 1e-3  # learning rate
weight_decay: float = 1e-4  # weight_decay
num_epochs: int = 15
train_set_split: float = 0.9
batch_size: int = 64
model_output_path = 'mlp_classifier.pt'
inference_output_path = '../../exploration/data/4_visium/4_visium_stofm_classification_domain_detection.csv'

In [22]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# define a simple MLP
class MLPClassifier(nn.Module):
    def __init__(self, d, num_classes, hidden=512, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, hidden),
            nn.GELU(),
            nn.Dropout(p),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# prepare dataset
class EmbDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.from_numpy(embeddings).float()
        self.labels = torch.from_numpy(labels).long()

    def __len__(self):
        return self.embeddings.shape[0]

    def __getitem__(self, i):
        return {'embedding': self.embeddings[i], 'label': self.labels[i]}


def prepare_dataset(embeddings: pd.DataFrame, labels: pd.Series,
                    train_set_split: float = 0.8, batch_size: int = 64):
    N = embeddings.shape[0]
    perm = np.random.permutation(N)
    split = int(train_set_split * N)
    train_idx, val_idx = perm[:split], perm[split:]

    train_ds = EmbDataset(embeddings[train_idx], labels[train_idx])
    val_ds = EmbDataset(embeddings[val_idx], labels[val_idx])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=0)
    return train_loader, val_loader


# evaluation
def evaluate(model, data_loader: DataLoader):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for data in data_loader:
            embedding = data['embedding'].to(device, non_blocking=True)
            label = data['label'].to(device, non_blocking=True)
            logits = model(embedding)
            pred = logits.argmax(1)
            all_pred.append(pred.numpy())
            all_true.append(label.numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return acc, macro_f1

In [52]:
# load dataset
import scanpy as sc
import numpy as np

adata = sc.read('../../exploration/data/1_visium/1_visium.h5ad')
labels = pd.Categorical(adata.obs['ground_truth']).codes
adata = sc.read('../../exploration/data/2_visium/2_visium.h5ad')
labels = np.concatenate(
    (labels, pd.Categorical(adata.obs['ground_truth']).codes), axis=0)
adata = sc.read('../../exploration/data/3_visium/3_visium.h5ad')
labels = np.concatenate(
    (labels, pd.Categorical(adata.obs['ground_truth']).codes), axis=0)

embeddings = np.concatenate(
    (np.load('../../exploration/data/1_visium/1_visium_stofm.npy'),
     np.load('../../exploration/data/2_visium/2_visium_stofm.npy'),
     np.load('../../exploration/data/3_visium/3_visium_stofm.npy')), axis=0)

# remove nan
mask = labels != -1
embeddings = embeddings[mask, :]
labels = labels[mask]

print(embeddings.shape)
print(labels.shape)
print(pd.Series(labels).unique())

(13390, 256)
(13390,)
[0 2 6 5 4 1 3]


In [53]:
# training
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPClassifier(input_emb_dim, num_classes, hidden=hidden_dim,
                      p=dropout).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                              weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                       factor=0.5, patience=15)

best_f1 = 0.0
for epoch in range(num_epochs):
    train_loader, val_loader = prepare_dataset(embeddings, labels,
                                               train_set_split, batch_size)
    for data in train_loader:
        embedding = data['embedding'].to(device, non_blocking=True)
        label = data['label'].to(device, non_blocking=True)
        logits = model(embedding)
        loss = criterion(logits, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    acc, f1 = evaluate(model, val_loader)
    scheduler.step(f1)
    print(f"Epoch {epoch:02d} | val acc={acc:.4f} | macroF1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), model_output_path)

Epoch 00 | val acc=0.6027 | macroF1=0.3966
Epoch 01 | val acc=0.6109 | macroF1=0.4258
Epoch 02 | val acc=0.6288 | macroF1=0.4759
Epoch 03 | val acc=0.6684 | macroF1=0.5099
Epoch 04 | val acc=0.6699 | macroF1=0.5449
Epoch 05 | val acc=0.6736 | macroF1=0.5840
Epoch 06 | val acc=0.6856 | macroF1=0.5414
Epoch 07 | val acc=0.6916 | macroF1=0.5674
Epoch 08 | val acc=0.6856 | macroF1=0.5530
Epoch 09 | val acc=0.6960 | macroF1=0.5828
Epoch 10 | val acc=0.6848 | macroF1=0.5769
Epoch 11 | val acc=0.6796 | macroF1=0.5464
Epoch 12 | val acc=0.7125 | macroF1=0.6406
Epoch 13 | val acc=0.7095 | macroF1=0.6197
Epoch 14 | val acc=0.6804 | macroF1=0.5980


## Inference

In [62]:
# load test dataset
import scanpy as sc
import numpy as np
import pandas as pd

adata = sc.read('../../exploration/data/4_visium/4_visium.h5ad')
labels = pd.Categorical(adata.obs['ground_truth']).codes
embeddings = np.load('../../exploration/data/4_visium/4_visium_stofm.npy')

# remove nan
mask = labels != -1
embeddings = embeddings[mask, :]
labels = labels[mask]

print(embeddings.shape)
print(labels.shape)
print(pd.Series(labels).unique())

(4595, 256)
(4595,)
[0 4 2 3 5 6 1]


In [63]:
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPClassifier(input_emb_dim, num_classes, hidden=hidden_dim,
                      p=dropout).to(device)
model.load_state_dict(torch.load(model_output_path, map_location=device))
model.eval()

with torch.no_grad():
    embeddings = torch.from_numpy(embeddings).float().to(device)  # [M, d]
    logits = model(embeddings)
    pred = logits.argmax(1).numpy()
    acc = accuracy_score(labels, pred)
    macro_f1 = f1_score(labels, pred, average="macro")

    print(f"Test acc={acc:.4f} | macroF1={macro_f1:.4f}")

pd.Series(pd.Categorical(pred)).to_csv(inference_output_path, index=False)

Test acc=0.6842 | macroF1=0.5820
